In [ ]:
import numpy as np
import sys
sys.path.append("/home/dajiang/smart-pixels-ml/two_bit_optimization_helpers")
from prepare_tfrecords import generate_tfrecords, load_tfrecords
from train import create_model, train
from input_digitization import process_inputs

##### Required arguments

In [ ]:
dataset_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets'
train_batch_size = 5000
val_batch_size = 5000
select_contained = True
two_bit_optimized = True # For part 2 only
noise = [0,80] 
timeslices = 2
labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'] # for slim model, use ['x-midplane','y-midplane','cotBeta'] instead
tfrecords_exist = True # if you already generated the TFRecords previously, set to True or you will redo it and overwrite the existing ones (saves time)
seed = 10

model_type='Conv2D_Max' # Conv2D_Max, Conv2D_Full, Mlp_Full, Mlp_Slim
train_type1='soft_quantize_layer' # full_precision, soft_quantize_layer, 2bit_optimized
train_type2='2bit_optimized' # full_precision, soft_quantize_layer, 2bit_optimized
soft_quantize_layer1=True # soft_quantize_layer
soft_quantize_layer2=False # 2bit_optimized
weights_directory='/data/dajiang/smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/'
epochs1=1
epochs2=11
initial_thresholds=[247.8, 668.4, 1662.9]
threshold_offset=80.0
initial_levels=np.array([0.0, 1.0, 2.0, 3.0], dtype=np.float32)

##### Generating and saving TFRecords and then loading the TFRecords

In [ ]:
dataset_train_dir1, dataset_validation_dir1, tfrecords_dir_train1, tfrecords_dir_val1 = generate_tfrecords(
    dataset_dir=dataset_dir,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    select_contained=select_contained,
    noise=noise,
    labels_list=labels_list,
    timeslices=timeslices,
    tfrecords_exist=tfrecords_exist
)

training_generator1, validation_generator1 = load_tfrecords(tfrecords_dir_train1, tfrecords_dir_val1)

##### Training part 1

In [ ]:
model1 = create_model(
    model_type=model_type,
    timeslices=timeslices,
    soft_quantize_layer=soft_quantize_layer1,
    initial_thresholds=initial_thresholds,
    threshold_offset=threshold_offset,
    initial_levels=initial_levels,
)

checkpoints_directory = train(
    model=model1,
    model_type=model_type, 
    weights_directory=weights_directory,
    training_generator=training_generator1,
    validation_generator=validation_generator1, 
    timeslices=timeslices,
    train_type=train_type1,
    epochs=epochs1,
    seed=seed, 
)

##### Input Digitization

In [ ]:
two_bit_train_dir, two_bit_test_dir = process_inputs(
    checkpoints=checkpoints_directory,
    dataset_train_dir=dataset_train_dir1,
    dataset_validation_dir=dataset_validation_dir1,
    model_type=model_type,
    timeslices=timeslices,
    initial_thresholds=initial_thresholds,
    threshold_offset=threshold_offset,
    initial_levels=initial_levels,
)

print(f'2bit train directory:{two_bit_train_dir}, 2bit test directory: {two_bit_test_dir}')

##### Generating and saving the TFRecords for the 2bit samples, then loading them for part 2 training

In [ ]:
# No noise for part 2 training
dataset_train_dir2, dataset_validation_dir2, tfrecords_dir_train2, tfrecords_dir_val2 = generate_tfrecords(
    dataset_dir=dataset_dir,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    select_contained=select_contained,
    two_bit_optimized=two_bit_optimized,
    labels_list=labels_list,
    timeslices=timeslices,
    tfrecords_exist=tfrecords_exist,
    model_type=model_type
)

training_generator2, validation_generator2 = load_tfrecords(tfrecords_dir_train2, tfrecords_dir_val2)

##### Training part 2

In [ ]:
model2 = create_model(
    model_type=model_type,
    timeslices=timeslices,
    soft_quantize_layer=soft_quantize_layer2,
)

checkpoints_directory = train(
    model=model2,
    model_type=model_type, 
    weights_directory=weights_directory,
    training_generator=training_generator2,
    validation_generator=validation_generator2, 
    timeslices=timeslices,
    train_type=train_type2,
    epochs=epochs2,
    seed=seed, 
)